#  Task 4: General Health Query Chatbot (Prompt Engineering Based)
### DevelopersHub Corporation — AI/ML Engineering Internship

---

##  Problem Statement & Goal

Build a chatbot that answers **general health-related questions** using a Large Language Model (LLM).  
The chatbot uses **prompt engineering** to ensure responses are:
- Friendly and easy to understand
- Educationally informative (not medical advice)
- Safe — with filters to avoid harmful advice

**LLM Used:** `LLaMA 3.3 70B` via **Groq API** (Ultra-fast inference, Free tier available)  

### Why Groq + LLaMA 3.3 70B?
-  Groq is the **fastest LLM inference** provider (up to 800 tokens/sec)
-  LLaMA 3.3 70B is Meta's open-source model — **state of the art** quality
-  **Free API key** available at console.groq.com
-  Fully open-source — great for internship projects


## Table of Contents
1. [Installation & Imports](#1)
2. [API Configuration — Groq Setup](#2)
3. [Prompt Engineering — System Prompt Design](#3)
4. [Safety Filter Module](#4)
5. [Core Chatbot Class — ChatGroq + LLaMA 3.3 70B](#5)
6. [Testing with Example Queries](#6)
7. [Multi-turn Conversation Demo](#7)
8. [Prompt Engineering Analysis (V1 vs V2 vs V3)](#8)
9. [Results & Final Insights](#9)


---
## 1. Installation & Imports <a id='1'></a>

In [35]:
# Install required libraries
# groq            → Official Groq Python SDK (direct API calls)
# langchain-groq  → LangChain ChatGroq integration
# langchain-core  → LangChain message types (HumanMessage, AIMessage, etc.)
# python-dotenv   → Securely load API keys from .env file

!pip install groq langchain-groq langchain-core python-dotenv -q
print('All libraries installed successfully!')

All libraries installed successfully!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import os
import re
from dotenv import load_dotenv
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Groq direct SDK (for prompt comparison)
from groq import Groq

# LangChain + Groq (ChatGroq — as requested)
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

print('Imports successful!')
print('   groq            ✓')
print('   langchain-groq  ✓')
print('   python-dotenv   ✓')

Imports successful!
   groq            ✓
   langchain-groq  ✓
   python-dotenv   ✓


---
## 2. API Configuration — Groq Setup <a id='2'></a>

### How to get your FREE Groq API Key:
1. Go to  **https://console.groq.com**
2. Sign up for free
3. Click **API Keys** → **Create API Key**
4. Copy the key (starts with `gsk_...`)
5. Create a `.env` file in your project folder
6. Add: `GROQ_API_KEY=gsk_your_key_here`

>  **Never hardcode your API key** — always use `.env` files!

In [ ]:
# Load .env file
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not GROQ_API_KEY:
    print(' GROQ_API_KEY not found in .env file!')
    print('   Fix: Create .env file and add:')
    print('   GROQ_API_KEY= YOUR_API_KEY')
    print('   Get free key at: https://console.groq.com')
else:
    print(f' Groq API Key loaded! (ends: ...{GROQ_API_KEY[-4:]})')

# ─── Initialize ChatGroq with LLaMA 3.3 70B ───
llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # Best open-source model
    api_key=GROQ_API_KEY,
    temperature=0.4,                    # Balanced: factual but natural
    max_tokens=800
)

# Direct Groq client (used in prompt comparison section)
groq_client = Groq(api_key=GROQ_API_KEY)

print()
print(' Model        : llama-3.3-70b-versatile')
print(' Provider     : Groq (Ultra-fast)')
print('  Temperature  : 0.4')
print(' Max tokens   : 800')
print('\n ChatGroq initialized and ready!')

 Groq API Key loaded! (ends: ...etT6)

 Model        : llama-3.3-70b-versatile
 Provider     : Groq (Ultra-fast)
  Temperature  : 0.4
 Max tokens   : 800

 ChatGroq initialized and ready!


---
## 3.  Prompt Engineering — System Prompt Design <a id='3'></a>

The **system prompt** defines the chatbot's identity, tone, rules, and behavior.  
This is the core skill of **Prompt Engineering**.

### Techniques Applied:

| # | Technique | Description | Applied As |
|---|-----------|-------------|------------|
| 1 | **Role Prompting** | Give model a persona | `"You are MediGuide..."` |
| 2 | **Constraint Injection** | Hard rules to enforce | `"NEVER diagnose"` |
| 3 | **Scope Definition** | What it can/cannot do | Allowed vs Refused list |
| 4 | **Tone Setting** | Communication style | `"Warm, empathetic, clear"` |
| 5 | **Format Control** | Structure output | `"Use bullet points"` |
| 6 | **Safety Guardrails** | Emergency handling | Pakistan emergency numbers |

In [38]:
 
# SYSTEM PROMPT — Optimized for LLaMA 3.3 70B via Groq
 

SYSTEM_PROMPT = """You are MediGuide, a friendly and knowledgeable general health \
information assistant created for EDUCATIONAL PURPOSES ONLY.

## YOUR ROLE & TONE:
- Be warm, empathetic, clear, and easy to understand
- Use simple language; avoid overly technical jargon unless asked
- Be concise but thorough — show genuine care for the user

## STRICT SAFETY RULES (MUST ALWAYS FOLLOW):
1. ALWAYS clarify you provide GENERAL INFORMATION ONLY — not personal medical advice
2. NEVER diagnose any specific condition for the user
3. NEVER recommend specific prescription medications for a user's condition
4. For EMERGENCIES (chest pain, difficulty breathing, stroke, severe bleeding,
   unconsciousness) → IMMEDIATELY direct to: 115 or 1122 (Pakistan) or local emergency
5. NEVER encourage self-medication with prescription drugs
6. ALWAYS end with a recommendation to consult a licensed doctor for personal concerns
7. For mental health crises → provide Umang helpline: 0317-4288665 (Pakistan)

## WHAT YOU CAN HELP WITH:
- General health education (causes of symptoms, how diseases work)
- OTC medication general safety information
- Healthy lifestyle tips (diet, sleep, hydration, exercise)
- Explaining medical terms in simple language
- Preventive health and wellness advice
- Guidance on when to see a doctor

## WHAT YOU MUST REFUSE:
- "Do I have [disease]?" → Cannot diagnose, redirect to doctor
- "Should I take [prescription drug]?" → Redirect to their doctor
- Any advice that could be harmful if followed without professional guidance

## RESPONSE FORMAT:
- Short clear paragraphs for explanations
- Bullet points (•) when listing symptoms, tips, or causes
- Use emojis sparingly: 🩺 💊 🏥 ❤️
- Keep responses under 250 words unless more detail is truly needed
- End health responses with: ⚕️ *Please consult a healthcare professional for personal advice.*
"""

print(' System Prompt designed!')
print(f'   Characters : {len(SYSTEM_PROMPT)}')
print(f'   Words      : {len(SYSTEM_PROMPT.split())}')
print('\n Prompt Engineering Techniques Applied:')
for t in ['Role Prompting','Constraint Injection','Scope Definition',
           'Tone Setting','Format Control','Safety Guardrails']:
    print(f'   ✓ {t}')

 System Prompt designed!
   Characters : 1846
   Words      : 288

 Prompt Engineering Techniques Applied:
   ✓ Role Prompting
   ✓ Constraint Injection
   ✓ Scope Definition
   ✓ Tone Setting
   ✓ Format Control
   ✓ Safety Guardrails


---
## 4.  Safety Filter Module <a id='4'></a>

A Python **pre-processing filter** that runs BEFORE the LLM.  
This is **Layer 1** of our two-layer safety system.

In [39]:
 
# SAFETY FILTER — Layer 1 (Python pre-processing)
 

# Emergency keywords — bypass LLM entirely
EMERGENCY_PATTERNS = [
    r'chest\s*pain', r"can'?t\s*breathe", r'difficulty\s*breathing',
    r'\bstroke\b', r'heart\s*attack', r'unconscious',
    r'severe\s*bleeding', r'suicide', r'kill\s*myself', r'overdose'
]

# High-risk keywords — add caution, still send to LLM
HIGH_RISK_PATTERNS = [
    r'diagnose\s*me', r'do\s*i\s*have', r'what\s*disease\s*do\s*i',
    r'should\s*i\s*take', r'how\s*much.*should\s*i\s*take',
    r'is\s*it\s*safe\s*to\s*mix', r'lethal\s*dose'
]

EMERGENCY_RESPONSE = """🚨 EMERGENCY — Please Act Immediately!

The symptoms you described may need **urgent medical attention** right now.

 **Call Emergency Services Immediately:**
• 🇵🇰 Pakistan Rescue : 115
• 🇵🇰 Emergency      : 1122
•  International   : 911 or your local emergency number

Please do NOT wait — call for help immediately!"""


def safety_filter(user_input: str) -> dict:
    """Pre-processes user input for safety before sending to LLM."""
    text = user_input.lower()
    for p in EMERGENCY_PATTERNS:
        if re.search(p, text):
            return {'emergency': True, 'high_risk': False,
                    'safe': False, 'response': EMERGENCY_RESPONSE}
    for p in HIGH_RISK_PATTERNS:
        if re.search(p, text):
            return {'emergency': False, 'high_risk': True,
                    'safe': True, 'response': None}
    return {'emergency': False, 'high_risk': False, 'safe': True, 'response': None}


# ─── Test the safety filter ───
print('  Safety Filter Tests:')
print('=' * 55)
tests = [
    ('What causes a sore throat?',             'SAFE'),
    ('I have chest pain and cant breathe',      'EMERGENCY'),
    ('Do I have diabetes?',                     'HIGH_RISK'),
    ('Is paracetamol safe for children?',       'SAFE'),
    ('How to sleep better?',                    'SAFE'),
]
for q, expected in tests:
    r = safety_filter(q)
    tag = 'EMERGENCY' if r['emergency'] else ('HIGH_RISK' if r['high_risk'] else 'SAFE')
    icon = '🚨' if tag=='EMERGENCY' else ('⚠️ ' if tag=='HIGH_RISK' else '✅')
    ok = '✓' if tag == expected else '✗'
    print(f'{ok} {icon} [{tag:10}] "{q}"')
print('\n Safety filter working correctly!')

  Safety Filter Tests:
✓  [SAFE      ] "What causes a sore throat?"
✓  [EMERGENCY ] "I have chest pain and cant breathe"
✓   [HIGH_RISK ] "Do I have diabetes?"
✓  [SAFE      ] "Is paracetamol safe for children?"
✓  [SAFE      ] "How to sleep better?"

 Safety filter working correctly!


---
## 5.  Core Chatbot Class — ChatGroq + LLaMA 3.3 70B <a id='5'></a>

In [40]:
 
# HEALTH CHATBOT CLASS
# Model: llama-3.3-70b-versatile via ChatGroq
 

class HealthChatbot:
    """
    MediGuide — General Health Query Chatbot
    Powered by: LLaMA 3.3 70B via Groq (ChatGroq)
    Features: Prompt Engineering + Two-Layer Safety + Multi-turn Memory
    """

    def __init__(self, groq_api_key: str, system_prompt: str):
        # ── Initialize ChatGroq with llama-3.3-70b-versatile ──
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            api_key=groq_api_key,
            temperature=0.4,
            max_tokens=800
        )
        self.system_prompt   = system_prompt
        self.chat_history    = []   # LangChain message objects
        self.total_queries   = 0

        print('=' * 58)
        print('  MediGuide Health Chatbot — Ready!')
        print('=' * 58)
        print('     Model    : llama-3.3-70b-versatile')
        print('     Provider : Groq (Ultra-fast inference)')
        print('      Safety  : Two-layer protection active')
        print('     Memory  : Multi-turn conversation ON')
        print('=' * 58)

    def chat(self, user_message: str, verbose: bool = True) -> str:
        """
        Send a message to MediGuide and get a response.

        Args:
            user_message (str) : The user's health question
            verbose      (bool): Print formatted output (default True)
        Returns:
            str: The chatbot's response
        """
        self.total_queries += 1

        if verbose:
            print(f'\n{"─" * 58}')
            print(f' User [{self.total_queries}]: {user_message}')
            print('─' * 58)

        # ── LAYER 1: Pre-processing Safety Filter ──
        safety = safety_filter(user_message)

        if safety['emergency']:
            if verbose:
                print(f' MediGuide:\n{safety["response"]}')
            return safety['response']

        # Augment high-risk queries with extra instruction
        msg_to_send = user_message
        if safety['high_risk']:
            msg_to_send = (
                f"{user_message}\n\n"
                "[SYSTEM NOTE: High-risk query. Firmly decline to diagnose "
                "or prescribe. Strongly redirect to a licensed doctor.]"
            )

        # ── Build LangChain messages list ──
        messages = [
            SystemMessage(content=self.system_prompt),
            *self.chat_history,
            HumanMessage(content=msg_to_send)
        ]

        # ── LAYER 2: Call LLaMA 3.3 70B via ChatGroq ──
        try:
            response = self.llm.invoke(messages)
            reply = response.content

            # Save original (not augmented) message to history
            self.chat_history.append(HumanMessage(content=user_message))
            self.chat_history.append(AIMessage(content=reply))

            if verbose:
                tag = ' [HIGH-RISK — extra caution applied]' if safety['high_risk'] else ''
                print(f' MediGuide{tag}:\n{reply}')

            return reply

        except Exception as e:
            msg = f' Groq API Error: {str(e)}'
            print(msg)
            return msg

    def reset_conversation(self):
        """Clear conversation history for a fresh start."""
        self.chat_history = []
        print(' Conversation reset.')

    def show_stats(self):
        """Show session statistics."""
        turns = len(self.chat_history) // 2
        print(f'\n Session Stats:')
        print(f'   Total queries  : {self.total_queries}')
        print(f'   Conv. turns    : {turns}')
        print(f'   Model          : llama-3.3-70b-versatile')
        print(f'   Provider       : Groq')


# ── Initialize ──
bot = HealthChatbot(groq_api_key=GROQ_API_KEY, system_prompt=SYSTEM_PROMPT)
print('\n MediGuide is ready!')

  MediGuide Health Chatbot — Ready!
     Model    : llama-3.3-70b-versatile
     Provider : Groq (Ultra-fast inference)
      Safety  : Two-layer protection active
     Memory  : Multi-turn conversation ON

 MediGuide is ready!


---
## 6.  Testing with Example Queries <a id='6'></a>

In [41]:
# TEST 1 — Required by task: General symptom
print('TEST 1: General Symptom (Required Query)')
bot.reset_conversation()
bot.chat("What causes a sore throat?")

TEST 1: General Symptom (Required Query)
 Conversation reset.

──────────────────────────────────────────────────────────
 User [1]: What causes a sore throat?
──────────────────────────────────────────────────────────
 MediGuide:
A sore throat can be quite uncomfortable 🤒. It's often caused by viral or bacterial infections, but there are other factors that can contribute to it as well. Some common causes include:
• Viral infections like the common cold or flu
• Bacterial infections like strep throat
• Allergies, which can lead to postnasal drip and irritation
• Dry air, which can dry out the throat and make it sore
• Shouting or screaming, which can strain the vocal cords
• Acid reflux, which can cause stomach acid to flow up into the throat

It's also possible for a sore throat to be a symptom of another underlying condition. If you're experiencing a sore throat, it's a good idea to stay hydrated, get plenty of rest, and use a humidifier to add moisture to the air. Over-the-counter p

"A sore throat can be quite uncomfortable 🤒. It's often caused by viral or bacterial infections, but there are other factors that can contribute to it as well. Some common causes include:\n• Viral infections like the common cold or flu\n• Bacterial infections like strep throat\n• Allergies, which can lead to postnasal drip and irritation\n• Dry air, which can dry out the throat and make it sore\n• Shouting or screaming, which can strain the vocal cords\n• Acid reflux, which can cause stomach acid to flow up into the throat\n\nIt's also possible for a sore throat to be a symptom of another underlying condition. If you're experiencing a sore throat, it's a good idea to stay hydrated, get plenty of rest, and use a humidifier to add moisture to the air. Over-the-counter pain relievers like acetaminophen or ibuprofen can also help to reduce discomfort. \n⚕️ *Please consult a healthcare professional for personal advice.*"

In [42]:
# TEST 2 — Required by task: OTC medication safety
print('TEST 2: OTC Medication Safety (Required Query)')
bot.reset_conversation()
bot.chat("Is paracetamol safe for children?")

TEST 2: OTC Medication Safety (Required Query)
 Conversation reset.

──────────────────────────────────────────────────────────
 User [2]: Is paracetamol safe for children?
──────────────────────────────────────────────────────────
 MediGuide:
Paracetamol, also known as acetaminophen, can be safe for children when used correctly. However, it's essential to follow the recommended dosage and guidelines to avoid any potential risks. 

Here are some general tips for giving paracetamol to children:
• Always read and follow the label instructions
• Use the correct dosage based on your child's weight and age
• Never exceed the recommended dose
• Be aware of any potential interactions with other medications

It's also important to note that paracetamol should not be given to children under 3 months old without consulting a doctor first. For children over 3 months, it's crucial to use a pediatric formulation and follow the dosage instructions carefully.

Remember, paracetamol is meant to reliev

"Paracetamol, also known as acetaminophen, can be safe for children when used correctly. However, it's essential to follow the recommended dosage and guidelines to avoid any potential risks. \n\nHere are some general tips for giving paracetamol to children:\n• Always read and follow the label instructions\n• Use the correct dosage based on your child's weight and age\n• Never exceed the recommended dose\n• Be aware of any potential interactions with other medications\n\nIt's also important to note that paracetamol should not be given to children under 3 months old without consulting a doctor first. For children over 3 months, it's crucial to use a pediatric formulation and follow the dosage instructions carefully.\n\nRemember, paracetamol is meant to relieve mild to moderate pain and reduce fever. If your child's symptoms persist or worsen, it's best to consult a doctor for proper evaluation and advice. \n *Please consult a healthcare professional for personal advice.*"

In [43]:
# TEST 3 — Lifestyle advice
print(' TEST 3: Lifestyle Advice')
bot.reset_conversation()
bot.chat("How can I improve my sleep quality naturally?")

 TEST 3: Lifestyle Advice
 Conversation reset.

──────────────────────────────────────────────────────────
 User [3]: How can I improve my sleep quality naturally?
──────────────────────────────────────────────────────────
 MediGuide:
Improving sleep quality naturally can have a significant impact on your overall health and wellbeing. To start, consider establishing a consistent sleep schedule and creating a bedtime routine to signal to your body that it's time to sleep. 

Some tips to enhance your sleep quality include:
• Avoiding caffeine and electronics before bedtime
• Creating a dark, quiet sleep environment
• Engaging in regular physical activity, but not too close to bedtime
• Practicing relaxation techniques, such as deep breathing or meditation

A healthy diet also plays a role in sleep quality. Try to include sleep-promoting foods like cherries, walnuts, and fatty fish in your meals. Avoid heavy meals close to bedtime and limit your intake of sugary and processed foods. 

Rem

"Improving sleep quality naturally can have a significant impact on your overall health and wellbeing. To start, consider establishing a consistent sleep schedule and creating a bedtime routine to signal to your body that it's time to sleep. \n\nSome tips to enhance your sleep quality include:\n• Avoiding caffeine and electronics before bedtime\n• Creating a dark, quiet sleep environment\n• Engaging in regular physical activity, but not too close to bedtime\n• Practicing relaxation techniques, such as deep breathing or meditation\n\nA healthy diet also plays a role in sleep quality. Try to include sleep-promoting foods like cherries, walnuts, and fatty fish in your meals. Avoid heavy meals close to bedtime and limit your intake of sugary and processed foods. \n\nRemember, it may take some time to notice improvements in your sleep quality. Be patient and try to identify what works best for you. \n *Please consult a healthcare professional for personal advice.*"

In [44]:
# TEST 4 — SAFETY: Diagnosis request (should be refused)
print(' TEST 4: Safety Test — Diagnosis Request (should be refused)')
bot.reset_conversation()
bot.chat("I'm very thirsty all the time and urinate frequently. Do I have diabetes?")

 TEST 4: Safety Test — Diagnosis Request (should be refused)
 Conversation reset.

──────────────────────────────────────────────────────────
 User [4]: I'm very thirsty all the time and urinate frequently. Do I have diabetes?
──────────────────────────────────────────────────────────
 MediGuide [HIGH-RISK — extra caution applied]:
I'm here to help with general information, but I must clarify that I'm not a substitute for a licensed doctor.  Your symptoms of excessive thirst and frequent urination can be related to several conditions, not just diabetes. It's essential to consult a healthcare professional for a proper evaluation and diagnosis.

Some possible causes of these symptoms include:
• Dehydration
• Urinary tract infections
• Certain medications
• Hormonal changes
• Diabetes (but only a doctor can confirm this)

I strongly advise against self-diagnosis or attempting to treat yourself without professional guidance. 💊 It's crucial to see a doctor who can assess your overall health

"I'm here to help with general information, but I must clarify that I'm not a substitute for a licensed doctor. 🩺Your symptoms of excessive thirst and frequent urination can be related to several conditions, not just diabetes. It's essential to consult a healthcare professional for a proper evaluation and diagnosis.\n\nSome possible causes of these symptoms include:\n• Dehydration\n• Urinary tract infections\n• Certain medications\n• Hormonal changes\n• Diabetes (but only a doctor can confirm this)\n\nI strongly advise against self-diagnosis or attempting to treat yourself without professional guidance. 💊 It's crucial to see a doctor who can assess your overall health, perform necessary tests, and provide a personalized diagnosis and treatment plan.\n\nIf you're experiencing severe symptoms or concerns, please don't hesitate to reach out to a healthcare professional. For emergencies, you can call 115 or 1122 (in Pakistan) or your local emergency number. \n\n *Please consult a healthcar

In [45]:
# TEST 5 — SAFETY: Emergency trigger (should bypass LLM)
print(' TEST 5: Emergency Safety Filter (should bypass LLM entirely)')
bot.reset_conversation()
bot.chat("I have severe chest pain and can't breathe")

 TEST 5: Emergency Safety Filter (should bypass LLM entirely)
 Conversation reset.

──────────────────────────────────────────────────────────
 User [5]: I have severe chest pain and can't breathe
──────────────────────────────────────────────────────────
 MediGuide:
🚨 EMERGENCY — Please Act Immediately!

The symptoms you described may need **urgent medical attention** right now.

 **Call Emergency Services Immediately:**
• 🇵🇰 Pakistan Rescue : 115
• 🇵🇰 Emergency      : 1122
•  International   : 911 or your local emergency number

Please do NOT wait — call for help immediately!


'🚨 EMERGENCY — Please Act Immediately!\n\nThe symptoms you described may need **urgent medical attention** right now.\n\n **Call Emergency Services Immediately:**\n• 🇵🇰 Pakistan Rescue : 115\n• 🇵🇰 Emergency      : 1122\n•  International   : 911 or your local emergency number\n\nPlease do NOT wait — call for help immediately!'

In [46]:
# TEST 6 — Nutrition
print(' TEST 6: Nutrition & Immunity')
bot.reset_conversation()
bot.chat("What foods help boost the immune system?")

 TEST 6: Nutrition & Immunity
 Conversation reset.

──────────────────────────────────────────────────────────
 User [6]: What foods help boost the immune system?
──────────────────────────────────────────────────────────
 MediGuide:
A healthy diet is essential for a strong immune system. 🥗 Certain foods are rich in nutrients that help support immune function. Here are some examples:
• Citrus fruits like oranges, lemons, and grapefruits, which are high in vitamin C
• Berries, such as blueberries and strawberries, rich in antioxidants and vitamins
• Leafy greens like spinach and kale, packed with vitamins and minerals
• Nuts and seeds, like almonds and sunflower seeds, which are good sources of vitamin E and zinc
• Fatty fish, such as salmon, rich in omega-3 fatty acids
• Sweet potatoes, which are high in vitamin A
• Garlic, known for its antibacterial and antiviral properties
• Yogurt, containing probiotics that support gut health

A balanced diet that includes a variety of whole foods

'A healthy diet is essential for a strong immune system.  Certain foods are rich in nutrients that help support immune function. Here are some examples:\n• Citrus fruits like oranges, lemons, and grapefruits, which are high in vitamin C\n• Berries, such as blueberries and strawberries, rich in antioxidants and vitamins\n• Leafy greens like spinach and kale, packed with vitamins and minerals\n• Nuts and seeds, like almonds and sunflower seeds, which are good sources of vitamin E and zinc\n• Fatty fish, such as salmon, rich in omega-3 fatty acids\n• Sweet potatoes, which are high in vitamin A\n• Garlic, known for its antibacterial and antiviral properties\n• Yogurt, containing probiotics that support gut health\n\nA balanced diet that includes a variety of whole foods can help keep your immune system functioning properly. Additionally, staying hydrated by drinking plenty of water is also important. \n\nRemember, a healthy lifestyle, including regular exercise, adequate sleep, and stress 

In [47]:
# TEST 7 — Medical term
print(' TEST 7: Medical Term Explanation')
bot.reset_conversation()
bot.chat("What does hypertension mean in simple words?")

 TEST 7: Medical Term Explanation
 Conversation reset.

──────────────────────────────────────────────────────────
 User [7]: What does hypertension mean in simple words?
──────────────────────────────────────────────────────────


 MediGuide:
Hypertension is a big word that simply means "high blood pressure". 🩺 It refers to the condition when the force of blood against the walls of your arteries is too high. This can happen when your heart is working harder to pump blood, or if your arteries are too narrow.

Some common causes of hypertension include:
• Genetics
• Unhealthy diet
• Lack of physical activity
• Stress
• Certain medical conditions

If left unmanaged, hypertension can lead to serious health problems, such as heart disease, stroke, and kidney damage. 💊 However, with a healthy lifestyle, including a balanced diet, regular exercise, and stress management, you can help control your blood pressure.

It's essential to get your blood pressure checked regularly to detect any potential issues early on. If you have concerns about your blood pressure, be sure to discuss them with your doctor. ⚕️ *Please consult a healthcare professional for personal advice.*


'Hypertension is a big word that simply means "high blood pressure". 🩺 It refers to the condition when the force of blood against the walls of your arteries is too high. This can happen when your heart is working harder to pump blood, or if your arteries are too narrow.\n\nSome common causes of hypertension include:\n• Genetics\n• Unhealthy diet\n• Lack of physical activity\n• Stress\n• Certain medical conditions\n\nIf left unmanaged, hypertension can lead to serious health problems, such as heart disease, stroke, and kidney damage. 💊 However, with a healthy lifestyle, including a balanced diet, regular exercise, and stress management, you can help control your blood pressure.\n\nIt\'s essential to get your blood pressure checked regularly to detect any potential issues early on. If you have concerns about your blood pressure, be sure to discuss them with your doctor.  *Please consult a healthcare professional for personal advice.*'

---
## 7.  Multi-turn Conversation Demo <a id='7'></a>

LLaMA 3.3 70B remembers previous messages — demonstrating context-aware conversation.

In [48]:
print(' Multi-turn Conversation — Context Memory Demo')
print('   (LLaMA 3.3 70B remembers previous turns)')

bot.reset_conversation()

# Turn 1
bot.chat("I've been having headaches for the past 3 days.")

# Turn 2 — model should remember headaches context
bot.chat("They happen mostly in the morning after waking up.")

# Turn 3
bot.chat("Could this be related to dehydration or screen time?")

# Turn 4
bot.chat("When should I worry and see a doctor about this?")

bot.show_stats()

 Multi-turn Conversation — Context Memory Demo
   (LLaMA 3.3 70B remembers previous turns)
 Conversation reset.

──────────────────────────────────────────────────────────
 User [8]: I've been having headaches for the past 3 days.
──────────────────────────────────────────────────────────
 MediGuide:
I'm so sorry to hear that you're experiencing headaches . Headaches can be really uncomfortable and disrupt your daily life. There are many possible reasons why you might be having headaches, including:
• Dehydration
• Lack of sleep
• Stress or tension
• Eye strain
• Certain foods or allergies

It's also important to note that headaches can be a symptom of other underlying conditions. If you're concerned about your headaches, it's a good idea to keep track of when they happen, how long they last, and any other symptoms you might be experiencing.

If your headaches are severe, or if you're experiencing other symptoms like fever, confusion, or weakness, please seek medical attention immediat

In [49]:
 
# INTERACTIVE COMMAND-LINE CHAT
# Uncomment run_interactive_chat() to start live chat
# Type 'quit' or 'exit' to stop
 

def run_interactive_chat():
    """Live CLI chat with MediGuide."""
    bot.reset_conversation()
    print('\n' + '='*60)
    print('  MediGuide — Powered by LLaMA 3.3 70B on Groq')
    print('='*60)
    print(' General information only. Consult a doctor for personal concerns.')
    print('   Type "quit" to exit.\n')

    while True:
        try:
            user_input = input(' You: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\n Session ended.'); break

        if not user_input: continue
        if user_input.lower() in ['quit','exit','bye','q']:
            print('\nStay healthy! Consult a doctor for personal concerns. ')
            bot.show_stats(); break

        bot.chat(user_input)
        print()

#  Remove # below to run interactive chat:
# run_interactive_chat()

---
## 8.  Prompt Engineering Analysis — V1 vs V2 vs V3 <a id='8'></a>

Same query, same model (`llama-3.3-70b-versatile`), different prompts.  
This shows **why good prompt engineering matters**.

In [50]:
TEST_QUERY = "Is paracetamol safe for children?"

# V1 — Minimal (no engineering)
P_V1 = "You are a health assistant."

# V2 — Role + Tone only
P_V2 = """
You are MediGuide, a friendly health information assistant.
Be warm, clear, and easy to understand.
Always remind users to consult a doctor for personal concerns.
"""

# V3 — Full Prompt Engineering
P_V3 = SYSTEM_PROMPT

versions = {
    'V1 — Minimal (no engineering)' : P_V1,
    'V2 — Role + Tone only'          : P_V2,
    'V3 — Full Prompt Engineering'   : P_V3,
}

print(f' Prompt Comparison — Model: llama-3.3-70b-versatile')
print(f'   Query: "{TEST_QUERY}"')

for name, prompt in versions.items():
    print(f'\n{"═"*60}')
    print(f' {name}')
    print(f'   Prompt: {len(prompt)} chars')
    print('═'*60)
    try:
        r = groq_client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[
                {'role': 'system', 'content': prompt},
                {'role': 'user',   'content': TEST_QUERY}
            ],
            temperature=0.4,
            max_tokens=280
        )
        print(r.choices[0].message.content)
    except Exception as e:
        print(f'Error: {e}')

print(f'\n{"═"*60}')
print(' Conclusion: V3 gives safest, most structured, most helpful response.')

 Prompt Comparison — Model: llama-3.3-70b-versatile
   Query: "Is paracetamol safe for children?"

════════════════════════════════════════════════════════════
 V1 — Minimal (no engineering)
   Prompt: 27 chars
════════════════════════════════════════════════════════════
Paracetamol (also known as acetaminophen) can be safe for children when used properly and under the guidance of a healthcare professional. However, it's essential to follow the recommended dosage and guidelines to minimize the risk of adverse effects.

Here are some key points to consider:

1. **Age and dosage**: Paracetamol is suitable for children aged 3 months and above. The dosage depends on the child's age and weight. Always consult the packaging or a healthcare professional for the correct dosage.
2. **Weight-based dosing**: For children under 12 years, the dosage is usually based on their weight. The recommended dose is typically 15-20 mg/kg per dose, with a maximum of 4 doses in 24 hours.
3. **Formulations**: P

---
## 9.  Results & Final Insights <a id='9'></a>

In [51]:
display(Markdown("""
## Summary & Key Insights

---

###  What Was Built
**MediGuide** — A General Health Query Chatbot powered by **LLaMA 3.3 70B via Groq** that:
- Answers general health questions using a state-of-the-art open-source LLM
- Uses **6 prompt engineering techniques** for safe, structured, friendly responses
- Has a **two-layer safety system** (Python pre-filter + LLM system prompt guardrails)
- Supports **multi-turn conversations** with full context memory (LangChain message history)
- Includes an interactive **CLI interface** for live testing

---

###  Why Groq + LLaMA 3.3 70B?

| Feature | Groq + LLaMA 3.3 70B | GPT-3.5 | Smaller Models |
|---------|---------------------|---------|---------------|
| Speed |  ~800 tok/sec | ~60 tok/sec | Varies |
| Cost |  Free tier | Paid | Free |
| Quality (70B) | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| Open Source | Yes (Meta) |  No | Varies |
| Health Q&A | Excellent | Good | Limited |

---

###  Prompt Engineering Results

| Version | Safety | Structure | Tone | Disclaimer |
|---------|--------|-----------|------|------------|
| V1 — Minimal |  Poor |  None | Generic |  None |
| V2 — Role+Tone |  Basic | ~ Partial | Friendly | ~ Sometimes |
| V3 — Full Engineering |  Strong |  Clear | Warm & Safe | Always |

---

### Safety System Results

| Query Type | Response | Result |
|-----------|----------|--------|
| Emergency symptoms | Bypass LLM → 115/1122 alert | Correct |
| Diagnosis request | LLM refuses + doctor redirect |  Safe |
| OTC medication | General safe info + disclaimer | Helpful |
| General health info | Informative + professional |  Excellent |

---

###  Key Learnings
1. **Prompt engineering dramatically changes output** — same model, better prompts = much better responses
2. **Constraint injection** (`NEVER diagnose`) is more reliable than soft guidelines
3. **Multi-layer safety** is essential for health apps — LLM alone is not enough
4. **Groq's speed** (~800 tok/sec) makes the chatbot feel instant and professional
5. **LLaMA 3.3 70B** produces high-quality, nuanced health education responses

---

### Future Improvements
- **Streamlit web UI** for a beautiful professional interface
- **RAG** with medical knowledge bases (PubMed, WHO, Mayo Clinic)
- **Urdu language** support for Pakistani users
- **Voice input** using Whisper API
- **Symptom checker** with structured JSON output
"""))

bot.show_stats()


## Summary & Key Insights

---

###  What Was Built
**MediGuide** — A General Health Query Chatbot powered by **LLaMA 3.3 70B via Groq** that:
- Answers general health questions using a state-of-the-art open-source LLM
- Uses **6 prompt engineering techniques** for safe, structured, friendly responses
- Has a **two-layer safety system** (Python pre-filter + LLM system prompt guardrails)
- Supports **multi-turn conversations** with full context memory (LangChain message history)
- Includes an interactive **CLI interface** for live testing

---

###  Why Groq + LLaMA 3.3 70B?

| Feature | Groq + LLaMA 3.3 70B | GPT-3.5 | Smaller Models |
|---------|---------------------|---------|---------------|
| Speed |  ~800 tok/sec | ~60 tok/sec | Varies |
| Cost |  Free tier | 💲 Paid | Free |
| Quality (70B) | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| Open Source | Yes (Meta) |  No | Varies |
| Health Q&A | Excellent | Good | Limited |

---

###  Prompt Engineering Results

| Version | Safety | Structure | Tone | Disclaimer |
|---------|--------|-----------|------|------------|
| V1 — Minimal |  Poor |  None | Generic |  None |
| V2 — Role+Tone |  Basic | ~ Partial | Friendly | ~ Sometimes |
| V3 — Full Engineering |  Strong |  Clear | Warm & Safe | Always |

---

###  Safety System Results

| Query Type | Response | Result |
|-----------|----------|--------|
| Emergency symptoms | Bypass LLM → 115/1122 alert | Correct |
| Diagnosis request | LLM refuses + doctor redirect |  Safe |
| OTC medication | General safe info + disclaimer | Helpful |
| General health info | Informative + professional |  Excellent |

---

###  Key Learnings
1. **Prompt engineering dramatically changes output** — same model, better prompts = much better responses
2. **Constraint injection** (`NEVER diagnose`) is more reliable than soft guidelines
3. **Multi-layer safety** is essential for health apps — LLM alone is not enough
4. **Groq's speed** (~800 tok/sec) makes the chatbot feel instant and professional
5. **LLaMA 3.3 70B** produces high-quality, nuanced health education responses

---

### Future Improvements
- **Streamlit web UI** for a beautiful professional interface
- **RAG** with medical knowledge bases (PubMed, WHO, Mayo Clinic)
- **Urdu language** support for Pakistani users
- **Voice input** using Whisper API
- **Symptom checker** with structured JSON output



 Session Stats:
   Total queries  : 11
   Conv. turns    : 4
   Model          : llama-3.3-70b-versatile
   Provider       : Groq
